In [51]:
%pip install geopy

Note: you may need to restart the kernel to use updated packages.


In [52]:
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from pathlib import Path
import time
from geopy.geocoders import Nominatim

¿Cómo ha evolucionado la accesibilidad económica a la vivienda en los municipios de Catalunya y cómo varían sus condiciones de vivienda, renta y contexto demográfico y territorial?

Metrica que ayude a comparar -> "Accesibilidad" comparada con el valor de la misma y el ingreso de las pesonas. 
Importante enteder que La ciudad de barcelona cuenta con una población en gran crecimiento que trabaja tanto para el mercado interno como los nomades digitales. Entonces hay que ver tema de informalidad.

## Analysis Universe

**Source:** Idescat — municipal population data for 2025.  
**Purpose:** Define the geographic scope of the analysis.  
**Selection:** Municipalities with more than 30,000 inhabitants in 2025.  
**Final universe:** 47 municipalities in Catalonia, used consistently across all datasets.

In [53]:
url = "https://api.idescat.cat/emex/v1/dades.json"

params = {
    "i": "f171",
    "tipus": "mun",
    "lang": "es"
}

response = requests.get(url, params=params)
response.raise_for_status()

data = response.json()

In [54]:
cols = data["fitxes"]["cols"]["col"]
values = data["fitxes"]["indicadors"]["i"]["v"].split(",")

df_population = pd.DataFrame({
    "municipality_code": [x["id"] for x in cols],
    "municipality": [x["content"] for x in cols],
    "population": pd.to_numeric(values)
})

In [55]:
# Preserve the original six-character Idescat identifier for API queries.
df_population["municipality_code_idescat"] = (
    df_population["municipality_code"].astype("string").str.strip().str.zfill(6)
)

# The canonical five-character key is used for municipal joins and exports.
df_population["municipality_code"] = (
    df_population["municipality_code_idescat"].str[:5]
)


In [56]:
analysis_universe = df_population[
    df_population["population"] > 30000
].copy()

In [57]:
analysis_universe = analysis_universe.sort_values(
    "population",
    ascending=False
).reset_index(drop=True)

## Municipality code standardization

**Canonical key:** `municipality_code` is a five-character string used for all municipal joins and CSV exports. 
The original six-character Idescat identifier is retained as `municipality_code_idescat` for API requests and traceability. 
Source codes are validated against the 47-municipality reference; unknown codes are reported rather than guessed. 
Catalonia-wide aggregate datasets such as annual contracts and mortgages do not have municipality keys and are not normalized.

In [58]:
# ============================================================
# MUNICIPALITY CODE STANDARDIZATION
# ============================================================

# Keep the original Idescat identifier for traceability/API requests.
analysis_universe["municipality_code_idescat"] = (
    analysis_universe["municipality_code_idescat"]
    .astype("string").str.strip().str.zfill(6)
)

# Canonical municipal key: five-digit code used by housing datasets.
analysis_universe["municipality_code"] = (
    analysis_universe["municipality_code_idescat"].str[:5]
)
analysis_universe["municipality_code_habitatge"] = (
    analysis_universe["municipality_code"]
)

# Build a reference from the 47 selected municipalities.
municipality_reference = analysis_universe[
    ["municipality", "municipality_code", "municipality_code_idescat"]
].copy()

assert len(municipality_reference) == 47
assert municipality_reference["municipality_code"].nunique() == 47
assert municipality_reference["municipality_code"].str.fullmatch(r"\d{5}").all()

valid_municipality_codes = set(municipality_reference["municipality_code"])
idescat_to_canonical = dict(zip(
    municipality_reference["municipality_code_idescat"],
    municipality_reference["municipality_code"]
))

def normalize_municipality_code(value):
    """Return a validated five-digit canonical code, or <NA> if unrecognized."""
    if pd.isna(value):
        return pd.NA
    code = str(value).strip()
    if code in idescat_to_canonical:
        return idescat_to_canonical[code]
    if code in valid_municipality_codes:
        return code
    return pd.NA

def normalize_municipality_codes(df, source_col="municipality_code"):
    """Copy a dataframe and standardize its municipality key safely."""
    result = df.copy()
    original = result[source_col].astype("string").str.strip()
    result["municipality_code_source"] = original
    result["municipality_code"] = original.map(normalize_municipality_code)

    unmatched = original.notna() & result["municipality_code"].isna()
    if unmatched.any():
        print(
            "REVIEW REQUIRED — unmatched municipality codes:",
            original[unmatched].drop_duplicates().tolist()[:30]
        )
    return result

print("Canonical municipality reference:", len(valid_municipality_codes), "municipalities")


Canonical municipality reference: 47 municipalities


## Population

**Source:** Idescat — annual municipal population data, 2013–2024.  
**Purpose:** Measure population size and provide demographic context for the housing market.  
**Cleaning:** Municipal and population fields were extracted, non-data rows removed, population converted to numeric, and year added.  
**Analysis:** Restricted to the 47 municipalities with more than 30,000 inhabitants in 2025, resulting in 564 municipality-year observations.  
**Validation:** Checked yearly coverage, missing municipality codes, and duplicated municipality-year combinations.

In [59]:
def extract_population_year(year):

    url = (
        f"https://www.idescat.cat/indicadors/"
        f"?id=aec&lang=es&n=15228&t={year}00"
    )

    df = pd.read_html(url)[0]

    df = df.iloc[:, [0, 1]].copy()

    df.columns = [
        "municipality",
        "population"
    ]

    df["population"] = (
        df["population"]
        .astype(str)
        .str.replace(".", "", regex=False)
    )

    df["population"] = pd.to_numeric(
        df["population"],
        errors="coerce"
    )

    df = df[
        df["population"].notna()
    ].copy()

    df["population"] = df["population"].astype(int)

    df["year"] = year

    return df

In [60]:
population_data = []

for year in range(2013, 2025):

    df_year = extract_population_year(year)

    population_data.append(df_year)

df_population_history = pd.concat(
    population_data,
    ignore_index=True
)

In [61]:
population_analysis = df_population_history[
    df_population_history["municipality"].isin(
        analysis_universe["municipality"]
    )
].copy()

population_analysis = population_analysis.reset_index(drop=True)

In [62]:
population_analysis = population_analysis.merge(
    analysis_universe[
        ["municipality", "municipality_code"]
    ],
    on="municipality",
    how="left"
)

In [63]:
population_analysis = population_analysis[
    [
        "municipality_code",
        "municipality",
        "year",
        "population"
    ]
].copy()

In [64]:
population_analysis.head()

,municipality_code,municipality,year,population
0,08019,Barcelona,2013,1611822
1,08101,"Hospitalet de Llobregat, l'",2013,254056
2,08015,Badalona,2013,219708
3,08279,Terrassa,2013,215055
4,08187,Sabadell,2013,207649


## Market

**Source:** Generalitat de Catalunya — registered housing sales data, 2013–2024.  
**Purpose:** Measure housing market activity and sale price levels at municipality level.  
**Cleaning:** Municipal codes and names were standardized, relevant sales and price variables were selected, non-municipal rows removed, and numeric fields converted to numeric format.  
**Analysis:** Restricted to the same 47 municipalities, with annual observations from 2013 to 2024.  
**Validation:** Checked yearly coverage, municipality-year duplicates, and coverage of the analysis universe.

In [65]:
sales_urls = {
    2013: "https://habitatge.gencat.cat/web/.content/home/dades/estadistiques/01_Estadistiques_de_construccio_i_mercat_immobiliari/02_Compravenda_i_preu_de_venda/02_Compravendes_d_habitatges_registrades_i_el_preu_de_venda/2013/MUN_anual_2013.xls",
    
    2014: "https://habitatge.gencat.cat/web/.content/home/dades/estadistiques/01_Estadistiques_de_construccio_i_mercat_immobiliari/02_Compravenda_i_preu_de_venda/02_Compravendes_d_habitatges_registrades_i_el_preu_de_venda/2014/MUN_anual_2014.xls",
    
    2015: "https://habitatge.gencat.cat/web/.content/home/dades/estadistiques/01_Estadistiques_de_construccio_i_mercat_immobiliari/02_Compravenda_i_preu_de_venda/02_Compravendes_d_habitatges_registrades_i_el_preu_de_venda/2015/MUN_anual_2015.xls",
    
    2016: "https://habitatge.gencat.cat/web/.content/home/dades/estadistiques/01_Estadistiques_de_construccio_i_mercat_immobiliari/02_Compravenda_i_preu_de_venda/02_Compravendes_d_habitatges_registrades_i_el_preu_de_venda/2016/MUN_anual_2016.xls",
    
    2017: "https://habitatge.gencat.cat/web/.content/home/dades/estadistiques/01_Estadistiques_de_construccio_i_mercat_immobiliari/02_Compravenda_i_preu_de_venda/02_Compravendes_d_habitatges_registrades_i_el_preu_de_venda/2017/MUN_acumulat_2017.xls",
    
    2018: "https://habitatge.gencat.cat/web/.content/home/dades/estadistiques/01_Estadistiques_de_construccio_i_mercat_immobiliari/02_Compravenda_i_preu_de_venda/02_Compravendes_d_habitatges_registrades_i_el_preu_de_venda/2018/MUN_acumulat_2018.xlsx",
    
    2019: "https://habitatge.gencat.cat/web/.content/home/dades/estadistiques/01_Estadistiques_de_construccio_i_mercat_immobiliari/02_Compravenda_i_preu_de_venda/02_Compravendes_d_habitatges_registrades_i_el_preu_de_venda/2019/MUN_acumulat_2019.xlsx",
    
    2020: "https://habitatge.gencat.cat/web/.content/home/dades/estadistiques/01_Estadistiques_de_construccio_i_mercat_immobiliari/02_Compravenda_i_preu_de_venda/02_Compravendes_d_habitatges_registrades_i_el_preu_de_venda/2020/MUN_acumulat_2020.xls",
    
    2021: "https://habitatge.gencat.cat/web/.content/home/dades/estadistiques/01_Estadistiques_de_construccio_i_mercat_immobiliari/02_Compravenda_i_preu_de_venda/02_Compravendes_d_habitatges_registrades_i_el_preu_de_venda/2021/MUN_acumulat_2021.xlsx",
    
    2022: "https://habitatge.gencat.cat/web/.content/home/dades/estadistiques/01_Estadistiques_de_construccio_i_mercat_immobiliari/02_Compravenda_i_preu_de_venda/02_Compravendes_d_habitatges_registrades_i_el_preu_de_venda/2022/MUN_acumulat_2022.xlsx",
    
    2023: "https://habitatge.gencat.cat/web/.content/home/dades/estadistiques/01_Estadistiques_de_construccio_i_mercat_immobiliari/02_Compravenda_i_preu_de_venda/02_Compravendes_d_habitatges_registrades_i_el_preu_de_venda/2023/MUN_acumulat_2023.xlsx",
    
    2024: "https://habitatge.gencat.cat/web/.content/home/dades/estadistiques/01_Estadistiques_de_construccio_i_mercat_immobiliari/02_Compravenda_i_preu_de_venda/02_Compravendes_d_habitatges_registrades_i_el_preu_de_venda/2024/MUN_acumulat_2024.xlsx"
}

In [66]:
import requests
from pathlib import Path

# Destination folder ->Choose your path
sales_folder = Path(
    r"C:\Users\jejoc\Desktop\Ironhack\Final Project\Sales"
)

# Create folder if it does not exist
sales_folder.mkdir(parents=True, exist_ok=True)

# Download files
for year, url in sales_urls.items():
    
    filename = url.split("/")[-1]
    filepath = sales_folder / filename
    
    response = requests.get(url)
    response.raise_for_status()
    
    with open(filepath, "wb") as file:
        file.write(response.content)
    
    print(f"{year} → {filename} ✓")

2013 → MUN_anual_2013.xls ✓
2014 → MUN_anual_2014.xls ✓
2015 → MUN_anual_2015.xls ✓
2016 → MUN_anual_2016.xls ✓
2017 → MUN_acumulat_2017.xls ✓
2018 → MUN_acumulat_2018.xlsx ✓
2019 → MUN_acumulat_2019.xlsx ✓
2020 → MUN_acumulat_2020.xls ✓
2021 → MUN_acumulat_2021.xlsx ✓
2022 → MUN_acumulat_2022.xlsx ✓
2023 → MUN_acumulat_2023.xlsx ✓
2024 → MUN_acumulat_2024.xlsx ✓


In [67]:
#Define column structure
sales_columns = [
    "municipality_code",
    "municipality",
    "sales_new_free",
    "sales_new_protected",
    "sales_used",
    "sales_total",
    "surface_new_free",
    "surface_new_protected",
    "surface_used",
    "surface_total",
    "price_new",
    "price_used",
    "price_total",
    "price_m2_new",
    "price_m2_used",
    "price_m2_total"
]

In [68]:
numeric_columns = [
    "sales_new_free",
    "sales_new_protected",
    "sales_used",
    "sales_total",
    "surface_new_free",
    "surface_new_protected",
    "surface_used",
    "surface_total",
    "price_new",
    "price_used",
    "price_total",
    "price_m2_new",
    "price_m2_used",
    "price_m2_total"
]

In [69]:
def clean_sales_data(df, year):
    
    # Extract municipal rows
    df = df.iloc[5:].copy()
    
    # Remove separator columns
    df = df.iloc[
        :, [0, 1, 2, 3, 4, 5, 7, 8, 9, 10, 12, 13, 14, 16, 17, 18]
    ]
    
    # Rename columns
    df.columns = sales_columns
    
    # Keep only individual municipalities
    df = df[df["municipality_code"].notna()].copy()
    
    # Validate and standardize against the canonical municipality reference.
    df = normalize_municipality_codes(df, "municipality_code")
    df = df[df["municipality_code"].notna()].copy()
    
    # Convert numerical variables
    df[numeric_columns] = df[numeric_columns].apply(
        pd.to_numeric
    )
    
    # Add year
    df["year"] = year
    
    return df

In [70]:
#Read and combine years
from pathlib import Path

sales_folder = Path(
    r"C:\Users\jejoc\Desktop\Ironhack\Final Project\Sales"
)

sales_files = {
    2013: "MUN_anual_2013.xls",
    2014: "MUN_anual_2014.xls",
    2015: "MUN_anual_2015.xls",
    2016: "MUN_anual_2016.xls",
    2017: "MUN_acumulat_2017.xls",
    2018: "MUN_acumulat_2018.xlsx",
    2019: "MUN_acumulat_2019.xlsx",
    2020: "MUN_acumulat_2020.xls",
    2021: "MUN_acumulat_2021.xlsx",
    2022: "MUN_acumulat_2022.xlsx",
    2023: "MUN_acumulat_2023.xlsx",
    2024: "MUN_acumulat_2024.xlsx"
}

In [71]:
sales_data = []

for year, filename in sales_files.items():
    
    filepath = sales_folder / filename
    
    df = pd.read_excel(
        filepath,
        header=None
    )
    
    df_clean = clean_sales_data(df, year)
    
    sales_data.append(df_clean)

df_sales = pd.concat(
    sales_data,
    ignore_index=True
)

sales_municipality_lookup = analysis_universe[
    ["municipality_code", "municipality"]
].copy()
analysis_sales_codes = set(sales_municipality_lookup["municipality_code"])
df_sales = df_sales.loc[
    df_sales["municipality_code"].isin(analysis_sales_codes)
].copy()
df_sales = df_sales.drop(columns="municipality").merge(
    sales_municipality_lookup,
    on="municipality_code",
    how="left",
    validate="many_to_one",
)

sales_municipality_count = df_sales["municipality_code"].nunique()
sales_name_count = df_sales["municipality"].nunique()
assert sales_municipality_count == sales_name_count == len(analysis_sales_codes) == 47
print(
    f"Sales analysis filtered to {sales_municipality_count} municipalities "
    "with standardized names."
)

REVIEW REQUIRED — unmatched municipality codes: ['08001', '25003', '43004', '25011', '43005', '08003', '25019', '25023', '43013', '08005', '43014', '17008', '43016', '17009', '08006', '08007', '08009', '08010', '08904', '25040', '17015', '08020', '08022', '08023', '17022', '25058', '08030', '17033', '08033', '08035']
REVIEW REQUIRED — unmatched municipality codes: ['08001', '25003', '43004', '25011', '43005', '08003', '25019', '25023', '43013', '08005', '43014', '17008', '43016', '17009', '08006', '08007', '08009', '08010', '08904', '25040', '17015', '08020', '08022', '08023', '17022', '25058', '08030', '17033', '08033', '08035']
REVIEW REQUIRED — unmatched municipality codes: ['08001', '25003', '43004', '25011', '43005', '08003', '25019', '25023', '43013', '08005', '43014', '17008', '43016', '17009', '08006', '08007', '08009', '08010', '08904', '25040', '17015', '08020', '08022', '08023', '17022', '25058', '08030', '17033', '08033', '08035']
REVIEW REQUIRED — unmatched municipality co

In [72]:
df_sales.tail()

,municipality_code,sales_new_free,sales_new_protected,sales_used,sales_total,surface_new_free,surface_new_protected,surface_used,surface_total,price_new,price_used,price_total,price_m2_new,price_m2_used,price_m2_total,municipality_code_source,year,municipality
559,43163,94,13,842,949,98.10,51.54,94.19,93.95,140.93895,146.69342,146.09129,1575.25,1724.99,1709.32,43163,2024,"Vendrell, el"
560,08298,242,4,312,558,90.46,77.25,86.76,88.35,241.69044,166.83788,201.98476,2663.06,1938.45,2278.69,08298,2024,Vic
561,08301,39,0,696,735,94.11,0.00,89.22,89.49,282.77647,231.27111,234.35818,3020.08,2687.68,2707.60,08301,2024,Viladecans
562,08305,168,6,301,475,85.48,89.00,86.20,85.97,189.58155,135.06963,157.54877,2216.59,1581.09,1843.15,08305,2024,Vilafranca del Penedès
563,08307,306,1,839,1146,78.35,57.00,88.48,85.57,289.83818,218.74230,239.77612,3733.19,2683.54,2994.08,08307,2024,Vilanova i la Geltrú


## Rental Market

**Source:** Generalitat de Catalunya / INCASÒL — rental contracts and average contractual rent data, 2013–2024.  
**Purpose:** Measure rental market conditions through the average monthly contractual rent at municipality level.  
**Cleaning:** Municipal codes and names were standardized, relevant fields were selected, non-municipal rows removed, and rent values converted to numeric format.  
**Analysis:** Restricted to the same 47 municipalities, with annual observations from 2013 to 2024.  
**Validation:** Checked yearly coverage, municipality-year duplicates, and coverage of the analysis universe.

In [73]:
# ============================================================
# 1. LOAD RENTAL EXCEL
# ============================================================

rent_file = Path(
    r"C:\Users\jejoc\Desktop\Ironhack\Final Project\municipis_trimestral_lloguer.xlsx"
)

xls_rent = pd.ExcelFile(rent_file)


# ============================================================
# 2. FUNCTION TO EXTRACT ANNUAL RENTAL DATA
# ============================================================

def extract_rent_year(df, year):
    
    # Header position changes from 2015 to 2016
    header_row = 4 if year <= 2015 else 5
    
    # Keep rows below the header
    df = df.iloc[header_row + 1:].copy()
    
    # Keep relevant columns
    df = df.iloc[:, [0, 1, 2, 3]].copy()
    
    # Rename columns
    df.columns = [
        "municipality_code",
        "municipality",
        "comarca",
        "rent_mean"
    ]
    
    # Remove rows without municipality code
    df = df[df["municipality_code"].notna()].copy()
    
    # Validate source codes against the canonical reference.
    df = normalize_municipality_codes(df, "municipality_code")
    df = df[df["municipality_code"].notna()].copy()
    
    # Convert rent to numeric
    df["rent_mean"] = pd.to_numeric(
        df["rent_mean"],
        errors="coerce"
    )
    
    # Add year
    df["year"] = year
    
    return df


# ============================================================
# 3. EXTRACT ALL YEARS
# ============================================================

rent_data = []

for sheet in xls_rent.sheet_names:
    
    year = int(sheet)
    
    df = pd.read_excel(
        rent_file,
        sheet_name=sheet,
        header=None
    )
    
    df_clean = extract_rent_year(
        df,
        year
    )
    
    rent_data.append(df_clean)


# Combine all years
df_rent = pd.concat(
    rent_data,
    ignore_index=True
)


# ============================================================
# 4. DEFINE ANALYSIS PERIOD
# ============================================================

analysis_years = list(range(2013, 2025))


# ============================================================
# 5. FILTER TO ANALYSIS UNIVERSE
# ============================================================

rent_analysis = df_rent[
    (df_rent["year"].isin(analysis_years)) &
    (
        df_rent["municipality_code"].isin(
            analysis_universe["municipality_code"]
        )
    )
].copy()


# ============================================================
# 6. RESET INDEX
# ============================================================

rent_analysis = rent_analysis.reset_index(drop=True)

REVIEW REQUIRED — unmatched municipality codes: ['25001', '08001', '25002', '25003', '08002', '17001', '08014', '43001', '17002', '25038', '25004', '25005', '25006', '17003', '25007', '25008', '25009', '43002', '43003', '17004', '43004', '25010', '25011', '25012', '43005', '43904', '43006', '43007', '08003', '43008']
REVIEW REQUIRED — unmatched municipality codes: ['25001', '08001', '25002', '25003', '08002', '17001', '08014', '43001', '17002', '25038', '25004', '25005', '25006', '17003', '25007', '25008', '25009', '43002', '43003', '17004', '43004', '25010', '25011', '25012', '43005', '43904', '43006', '43007', '08003', '43008']
REVIEW REQUIRED — unmatched municipality codes: ['25001', '08001', '25002', '25003', '08002', '17001', '08014', '43001', '17002', '25038', '25004', '25005', '25006', '17003', '25007', '25008', '25009', '43002', '43003', '17004', '43004', '25010', '25011', '25012', '43005', '43904', '43006', '43007', '08003', '43008']
REVIEW REQUIRED — unmatched municipality co

In [74]:
rent_analysis.head()

,municipality_code,municipality,comarca,rent_mean,municipality_code_source,year
0,08015,Badalona,Barcelonès,883.348102,08015,2024
1,08252,Barberà del Vallès,Vallès Occidental,802.903994,08252,2024
2,08019,Barcelona,Barcelonès,1147.222120,08019,2024
3,17023,Blanes,Selva,618.787797,17023,2024
4,43037,Calafell,Baix Penedès,762.565625,43037,2024


## Income

**Source:** Idescat — Gross Disposable Household Income (RFDB), 2013–2023.  
**Purpose:** Measure the economic context of each municipality and support the analysis of housing affordability.  
**Cleaning:** Municipality identifiers were extracted, years standardized, and income variables converted to numeric.  
**Analysis:** Restricted to the same 47 municipalities; income per capita is the main variable, complemented by the Catalonia income index.  
**Validation:** Checked yearly coverage, missing municipality codes, and duplicated municipality-year combinations. The 2023 observation is provisional according to the source.

In [75]:
# Import RAW TXT
df_income_raw = pd.read_csv(
    r"C:\Users\jejoc\Desktop\Ironhack\Final Project\renta_bruta_anual.txt",
    sep="\t",
    skiprows=3,
    header=None
)

# Assign column names
df_income_raw.columns = [
    "municipality_raw",
    "year",
    "income_thousand_eur",
    "income_per_capita_eur",
    "income_index_catalonia"
]

# Separate municipality code and name
df_income_raw[["municipality_code", "municipality"]] = (
    df_income_raw["municipality_raw"]
    .str.extract(r"\[(\d+)\]\s*(.*)")
)

# Remove "(p)" from 2023 and convert year to integer
df_income_raw["year"] = (
    df_income_raw["year"]
    .str.replace(r"\s*\(p\)", "", regex=True)
    .astype(int)
)

# Normalize municipality identifiers against the validated reference.
df_income_raw = normalize_municipality_codes(df_income_raw, "municipality_code")

# Remove original combined municipality column
df_income_raw = df_income_raw.drop(columns="municipality_raw")

# Reorder columns
df_income_raw = df_income_raw[
    [
        "municipality_code",
        "municipality",
        "year",
        "income_thousand_eur",
        "income_per_capita_eur",
        "income_index_catalonia"
    ]
]

df_income_raw.sample(10)

REVIEW REQUIRED — unmatched municipality codes: ['080018', '250030', '080142', '250387', '250077', '250083', '430022', '430043', '250117', '250122', '430056', '439044', '080039', '250138', '430094', '250169', '250194', '250215', '430115', '170062', '250236', '430120', '170078', '430136', '080057', '439060', '430141', '170084', '250273', '250292']


,municipality_code,municipality,year,income_thousand_eur,income_per_capita_eur,income_index_catalonia
3156,<NA>,Puigpelat,2013,14293.0,13234.0,827.0
699,<NA>,Bellvís,2017,31463.0,13771.0,788.0
353,<NA>,Arenys de Mar,2022,319925.0,19766.0,1032.0
2578,<NA>,Naut Aran,2019,32332.0,18000.0,975.0
1031,<NA>,Calldetenes,2015,39959.0,16738.0,992.0
3212,<NA>,Riba-roja d'Ebre,2023,19764.0,17413.0,838.0
2004,<NA>,Hostalric,2021,71326.0,16412.0,881.0
3459,08194,Sant Adrià de Besòs,2018,544435.0,14856.0,839.0
5113,<NA>,Vilassar de Dalt,2014,167198.0,18998.0,1173.0
491,08015,Badalona,2016,3442867.0,16133.0,938.0


In [76]:
income_analysis = df_income_raw[
    (df_income_raw["year"].isin(range(2013, 2024))) &
    (
        df_income_raw["municipality_code"].isin(
            analysis_universe["municipality_code"]
        )
    )
].copy()

income_analysis = income_analysis.reset_index(drop=True)

## Demography

**Source:** Idescat — Census of Population and Housing, indicators of age and sex, 2021–2025.  
**Purpose:** Describe the demographic structure and age profile of the municipalities included in the analysis.  
**Cleaning:** Municipal codes were used as identifiers, the selected age indicators were extracted, and municipality names were added from the analysis universe.  
**Analysis:** Restricted to the same 47 municipalities, with annual observations from 2021 to 2025. The main variables include mean age, median age, and the population shares aged 0–15, 16–64, and 65 or over.  
**Validation:** Checked municipality coverage, missing values, and the consistency of age-group proportions.

In [77]:
age_data = []

url = "https://api.idescat.cat/taules/v2/censph/16400/19880/mun/data"

for year in range(2021, 2026):

    for municipality_code in analysis_universe["municipality_code_idescat"]:

        params = {
            "YEAR": str(year),
            "MUN": municipality_code,
            "CONCEPT": "AGE_MEAN,AGE_MED,PP_Y_GE65,PP_Y00_15,OPP_Y16_64",
            "SEX": "TOTAL"
        }

        response = requests.get(url, params=params)
        response.raise_for_status()

        data = response.json()

        concepts_returned = (
            data["dimension"]["CONCEPT"]["category"]["index"]
        )

        values = data["value"]

        row = {
            "municipality_code": normalize_municipality_code(municipality_code),
            "year": year
        }

        for concept, value in zip(concepts_returned, values):
            row[concept] = value

        age_data.append(row)

df_age = pd.DataFrame(age_data)

In [78]:
df_age.head()

,municipality_code,year,PP_Y00_15,OPP_Y16_64,PP_Y_GE65,AGE_MEAN,AGE_MED
0,08019,2021,12.98,65.67,21.35,44.63,44.0
1,08101,2021,14.84,65.16,20.00,43.35,43.3
2,08279,2021,17.59,65.30,17.11,41.73,42.6
3,08015,2021,16.21,64.50,19.29,42.93,43.8
4,08187,2021,16.64,64.61,18.75,42.65,43.4


## Economic Activity

**Source:** Idescat — Census of Population and Housing, population aged 16 or over by relationship with economic activity, 2021–2025.  
**Purpose:** Provide labour-market context for the municipalities and complement the analysis of housing affordability.  
**Cleaning:** Municipal codes were used as identifiers, the selected economic activity indicators were extracted, and municipality names were added from the analysis universe.  
**Analysis:** Restricted to the same 47 municipalities, with annual observations from 2021 to 2025. The selected indicators are employed population, unemployed population, and total active population.  
**Validation:** Checked municipality coverage, missing values, and duplicated municipality-year combinations.

In [79]:
url = "https://api.idescat.cat/taules/v2/censph/270/20184/mun/data"

activity_data = []

for year in range(2021, 2026):

    for municipality_code in analysis_universe["municipality_code_idescat"]:

        params = {
            "YEAR": str(year),
            "MUN": municipality_code,
            "R_ECON_ACT": "PO_EMP,PO_UNE,PO_ACTT",
            "CONCEPT": "POP_YGE16T"
        }

        response = requests.get(url, params=params)
        response.raise_for_status()

        data = response.json()

        concepts_returned = (
            data["dimension"]["R_ECON_ACT"]["category"]["index"]
        )

        values = data["value"]

        row = {
            "municipality_code": normalize_municipality_code(municipality_code),
            "year": year
        }

        for concept, value in zip(concepts_returned, values):
            row[concept] = value

        activity_data.append(row)

df_activity = pd.DataFrame(activity_data)


In [80]:
df_activity.head()

,municipality_code,year,PO_EMP,PO_UNE,PO_ACTT
0,08019,2021,680147.0,85907.0,766054.0
1,08101,2021,105192.0,17674.0,122866.0
2,08279,2021,92252.0,16028.0,108280.0
3,08015,2021,88146.0,15586.0,103732.0
4,08187,2021,88718.0,14366.0,103084.0


## Annual Rental Contracts

**Source:** Idescat quarterly rental-contract statistics for Catalonia.  
**Purpose:** Track the annual number of signed rental contracts.  
**Processing:** Select the quarterly contract-count indicator and sum its values by year. The number of reported quarters is retained to identify incomplete years; 2026 currently contains only Q1.

In [81]:
contratos_raw = pd.read_csv(
    r"C:\Users\jejoc\Desktop\Ironhack\Final Project\contratos_de_alquiler.csv",
    sep=";",
    decimal=","
)

contratos_raw[["quarter", "year"]] = contratos_raw["row"].str.extract(
    r"T(\d)/(\d{4})"
).astype(int)

contracts_by_year = (
    contratos_raw[
        contratos_raw["col"] == "Número de contratos"
    ]
    .groupby("year")
    .agg(
        contracts=("value", "sum"),
        quarters_reported=("quarter", "nunique")
    )
    .reset_index()
    .sort_values("year")
)

contracts_by_year["year_complete"] = (
    contracts_by_year["quarters_reported"] == 4
)


print("Annual rental contracts:")
display(contracts_by_year)

Annual rental contracts:


,year,contracts,quarters_reported,year_complete
0,2005,52941.0,4,True
1,2006,56812.0,4,True
2,2007,62671.0,4,True
3,2008,80126.0,4,True
4,2009,97818.0,4,True
5,2010,116188.0,4,True
6,2011,127813.0,4,True
7,2012,138621.0,4,True
8,2013,150200.0,4,True
9,2014,149733.0,4,True


## Annual Mortgage Counts

**Source:** Idescat annual mortgage statistics for Catalonia.  
**Purpose:** Track the yearly number of mortgages constituted.  
**Processing:** Select the “Mortgages constituted (thousands)” indicator and convert it to an estimated number of mortgages. Provisional source years are flagged.

In [82]:
hipotecas_raw = pd.read_csv(
    r"C:\Users\jejoc\Desktop\Ironhack\Final Project\hipotecas.csv"
)

# Extraer cantidad de hipotecas
mortgages_by_year = (
    hipotecas_raw[
        hipotecas_raw["col"] == "Hipotecas constituidas (miles)"
    ]
    .assign(
        year=lambda df: df["row"].str.extract(r"(\d{4})")[0].astype(int),
        provisional=lambda df: df["row"].str.contains("(p)", regex=False)
    )
    .rename(columns={"value": "mortgages_thousands"})
    [["year", "mortgages_thousands", "provisional"]]
    .copy()
)

# Extraer capital prestado
capital_by_year = (
    hipotecas_raw[
        hipotecas_raw["col"] == "Capital prestado (miles M€)"
    ]
    .assign(
        year=lambda df: df["row"].str.extract(r"(\d{4})")[0].astype(int),
        provisional=lambda df: df["row"].str.contains("(p)", regex=False)
    )
    .rename(columns={"value": "capital_billion_eur"})
    [["year", "capital_billion_eur", "provisional"]]
    .copy()
)

# Unir ambos indicadores por año
mortgages_by_year = (
    mortgages_by_year
    .merge(
        capital_by_year[["year", "capital_billion_eur"]],
        on="year",
        how="outer",
        validate="one_to_one"
    )
    .sort_values("year")
    .reset_index(drop=True)
)

# Convertir cantidad de hipotecas de miles a unidades
mortgages_by_year["mortgages_estimated"] = (
    mortgages_by_year["mortgages_thousands"] * 1000
).round().astype("Int64")

# Filtrar el periodo del proyecto
mortgages_by_year = mortgages_by_year[
    mortgages_by_year["year"].between(2013, 2024)
].reset_index(drop=True)


display(mortgages_by_year)

,year,mortgages_thousands,provisional,capital_billion_eur,mortgages_estimated
0,2013,45.2,False,5.9,45200
1,2014,43.6,False,6.2,43600
2,2015,54.3,False,8.1,54300
3,2016,60.2,False,9.6,60200
4,2017,67.0,False,10.8,67000
5,2018,75.3,False,12.5,75300
6,2019,75.0,False,13.3,75000
7,2020,71.9,False,14.0,71900
8,2021,88.7,False,16.0,88700
9,2022,101.2,False,19.3,101200


# Distance

In [83]:
# Preparar listado de municipios para la extracción geográfica
municipalities_geo = (
    analysis_universe[
        ["municipality_code", "municipality"]
    ]
    .drop_duplicates()
    .sort_values("municipality_code")
    .reset_index(drop=True)
)

# Validaciones
assert municipalities_geo["municipality_code"].nunique() == 47
assert municipalities_geo["municipality"].notna().all()
assert municipalities_geo["municipality_code"].notna().all()

display(municipalities_geo.head())
print("Municipios:", len(municipalities_geo))

,municipality_code,municipality
0,08015,Badalona
1,08019,Barcelona
2,08056,Castelldefels
3,08073,Cornellà de Llobregat
4,08077,Esplugues de Llobregat


Municipios: 47


In [84]:
geolocator = Nominatim(
    user_agent="catalunya_housing_accessibility_project"
)

coordinates = []

for _, row in municipalities_geo.iterrows():
    municipality = row["municipality"]

    try:
        location = geolocator.geocode(
            f"{municipality}, Catalunya, Spain",
            exactly_one=True,
            country_codes="es"
        )

        if location:
            coordinates.append({
                "municipality_code": row["municipality_code"],
                "municipality": municipality,
                "latitude": location.latitude,
                "longitude": location.longitude
            })
        else:
            coordinates.append({
                "municipality_code": row["municipality_code"],
                "municipality": municipality,
                "latitude": None,
                "longitude": None
            })

    except Exception as e:
        print(f"Error con {municipality}: {e}")
        coordinates.append({
            "municipality_code": row["municipality_code"],
            "municipality": municipality,
            "latitude": None,
            "longitude": None
        })

    time.sleep(1.1)

coordinates_df = pd.DataFrame(coordinates)


display(coordinates_df)

,municipality_code,municipality,latitude,longitude
0,08015,Badalona,41.449354,2.248254
1,08019,Barcelona,41.382580,2.177073
2,08056,Castelldefels,41.286102,1.982417
3,08073,Cornellà de Llobregat,41.355724,2.070622
4,08077,Esplugues de Llobregat,41.377809,2.088626
5,08089,Gavà,41.305093,2.006313
6,08096,Granollers,41.607956,2.287601
7,08101,"Hospitalet de Llobregat, l'",41.359512,2.099632
8,08102,Igualada,41.579018,1.617346
9,08113,Manresa,41.728894,1.828677


In [85]:

# Ciudades de referencia: longitud, latitud
reference_cities = {
    "barcelona": (2.177073, 41.382580),
    "girona": (2.819944, 41.979301),
    "tarragona": (1.254606, 41.117236),
    "lleida": (0.626784, 41.614761)
}

distances = []

for _, row in coordinates_df.iterrows():
    result_row = {
        "municipality_code": row["municipality_code"],
        "municipality": row["municipality"]
    }

    for city, (ref_lon, ref_lat) in reference_cities.items():
        # Distancia cero si el municipio es la propia ciudad de referencia
        if row["municipality"].casefold() == city:
            result_row[f"distance_to_{city}_km"] = 0.0
            continue

        coordinates = (
            f"{ref_lon},{ref_lat};"
            f"{row['longitude']},{row['latitude']}"
        )

        url = f"https://router.project-osrm.org/route/v1/driving/{coordinates}"

        try:
            response = requests.get(
                url,
                params={"overview": "false"},
                timeout=30
            )
            response.raise_for_status()
            data = response.json()

            if data["code"] == "Ok" and data["routes"]:
                result_row[f"distance_to_{city}_km"] = (
                    data["routes"][0]["distance"] / 1000
                )
            else:
                result_row[f"distance_to_{city}_km"] = None

        except Exception as e:
            print(f"Error: {row['municipality']} → {city}: {e}")
            result_row[f"distance_to_{city}_km"] = None

        time.sleep(1)

    distances.append(result_row)

distance_analysis = pd.DataFrame(distances)

In [86]:
from pathlib import Path

# ============================================================
# VALIDATE MUNICIPALITY DATASETS BEFORE EXPORT
# ============================================================

municipal_datasets = {
    "analysis_universe": analysis_universe,
    "population_analysis": population_analysis,
    "sales_analysis": df_sales,
    "rent_analysis": rent_analysis,
    "income_analysis": income_analysis,
    "demographic_analysis": demographic_analysis if "demographic_analysis" in globals() else df_age,
    "activity_analysis": activity_analysis if "activity_analysis" in globals() else df_activity,
    "distance_analysis": distance_analysis,
}

for name, dataset in municipal_datasets.items():
    assert "municipality_code" in dataset.columns, f"{name}: missing municipality_code"
    codes = dataset["municipality_code"].astype("string").str.strip()
    invalid = codes.isna() | ~codes.str.fullmatch(r"\\d{5}", na=False)
    if invalid.any():
        print(f"REVIEW REQUIRED — {name}: invalid codes")
        display(dataset.loc[invalid].head(10))
    if "year" in dataset.columns:
        duplicates = dataset.duplicated(["municipality_code", "year"], keep=False)
        if duplicates.any():
            print(f"REVIEW REQUIRED — {name}: duplicate municipality-year keys")
            display(dataset.loc[duplicates].head(10))
    print(f"{name}: rows={len(dataset)}, municipalities={codes.nunique()}, invalid={int(invalid.sum())}")

assert analysis_universe["municipality_code"].nunique() == 47

# ============================================================
# CREATE CLEAN DATA FOLDER
# ============================================================

output_folder = Path(r"C:\Users\jejoc\Desktop\Ironhack\Final Project\Clean_data")
output_folder.mkdir(exist_ok=True)


# ============================================================
# 1. ANALYSIS UNIVERSE
# ============================================================

analysis_universe.to_csv(
    output_folder / "analysis_universe.csv",
    index=False
)


# ============================================================
# 2. POPULATION
# ============================================================

population_analysis.to_csv(
    output_folder / "population_analysis.csv",
    index=False
)


# ============================================================
# 3. HOME SALES
# ============================================================

df_sales.to_csv(
    output_folder / "sales_analysis.csv",
    index=False
)


# ============================================================
# 4. RENTAL MARKET
# ============================================================

rent_analysis.to_csv(
    output_folder / "rent_analysis.csv",
    index=False
)


# ============================================================
# 5. INCOME
# ============================================================

income_analysis.to_csv(
    output_folder / "income_analysis.csv",
    index=False
)


# ============================================================
# 6. DEMOGRAPHY
# ============================================================

demographic_analysis = df_age.merge(
    analysis_universe[
        ["municipality_code", "municipality"]
    ],
    on="municipality_code",
    how="left"
)

demographic_analysis = demographic_analysis[
    [
        "municipality_code",
        "municipality",
        "year",
        "AGE_MEAN",
        "AGE_MED",
        "PP_Y00_15",
        "OPP_Y16_64",
        "PP_Y_GE65"
    ]
].copy()

demographic_analysis = demographic_analysis.sort_values(
    ["municipality", "year"]
).reset_index(drop=True)

demographic_analysis.to_csv(
    output_folder / "demographic_analysis.csv",
    index=False
)


# ============================================================
# 7. ECONOMIC ACTIVITY
# ============================================================

activity_analysis = df_activity.merge(
    analysis_universe[
        ["municipality_code", "municipality"]
    ],
    on="municipality_code",
    how="left"
)

activity_analysis = activity_analysis[
    [
        "municipality_code",
        "municipality",
        "year",
        "PO_EMP",
        "PO_UNE",
        "PO_ACTT"
    ]
].copy()

activity_analysis = activity_analysis.sort_values(
    ["municipality", "year"]
).reset_index(drop=True)

activity_analysis.to_csv(
    output_folder / "activity_analysis.csv",
    index=False
)


# ============================================================
# 8. RENTAL CONTRACTS
# ============================================================
contracts_by_year.to_csv(
    output_folder / "contracts_by_year.csv",
    index=False
)

# ============================================================
# 9. MORTGAGES
# ============================================================
mortgages_by_year.to_csv(
    output_folder / "mortgages_by_year.csv",
    index=False
)

# ============================================================
# 10. DISTANCE TO BARCELONA
# ============================================================

distance_analysis.to_csv(
    output_folder / "distance_analysis.csv",
    index=False
)

# ============================================================
# CHECK EXPORTED FILES
# ============================================================

print("Datasets exportados correctamente:\n")

for filename in municipal_export_files:
    exported = pd.read_csv(
        output_folder / filename,
        dtype={"municipality_code": "string"}
    )

    codes = (
        exported["municipality_code"]
        .astype("string")
        .str.strip()
    )

    invalid = codes[
        codes.isna()
        | ~codes.str.fullmatch(r"\d{5}", na=False)
    ]

    print(
        f"{filename}: {len(invalid)} invalid codes "
        f"out of {len(codes)} rows"
    )

    assert invalid.empty, (
        f"{filename}: invalid exported codes "
        f"{invalid.unique()[:10]}"
    )


REVIEW REQUIRED — analysis_universe: invalid codes


,municipality_code,municipality,population,municipality_code_idescat,municipality_code_habitatge
0,08019,Barcelona,1713247,080193,08019
1,08101,"Hospitalet de Llobregat, l'",292161,081017,08101
2,08279,Terrassa,232676,082798,08279
3,08015,Badalona,230642,080155,08015
4,08187,Sabadell,224589,081878,08187
5,25120,Lleida,147369,251207,25120
6,43148,Tarragona,143260,431482,43148
7,08121,Mataró,131370,081213,08121
8,08245,Santa Coloma de Gramenet,123250,082457,08245
9,43123,Reus,111911,431233,43123


analysis_universe: rows=47, municipalities=47, invalid=47
REVIEW REQUIRED — population_analysis: invalid codes


,municipality_code,municipality,year,population
0,08019,Barcelona,2013,1611822
1,08101,"Hospitalet de Llobregat, l'",2013,254056
2,08015,Badalona,2013,219708
3,08279,Terrassa,2013,215055
4,08187,Sabadell,2013,207649
5,25120,Lleida,2013,139809
6,43148,Tarragona,2013,133545
7,08121,Mataró,2013,124099
8,08245,Santa Coloma de Gramenet,2013,120029
9,43123,Reus,2013,106790


population_analysis: rows=564, municipalities=47, invalid=564
REVIEW REQUIRED — sales_analysis: invalid codes


...

demographic_analysis: rows=235, municipalities=47, invalid=235
REVIEW REQUIRED — activity_analysis: invalid codes


,municipality_code,municipality,year,PO_EMP,PO_UNE,PO_ACTT
0,08015,Badalona,2021,88146.0,15586.0,103732.0
1,08015,Badalona,2022,91796.0,12476.0,104272.0
2,08015,Badalona,2023,94130.0,11585.0,105715.0
3,08015,Badalona,2024,96077.0,11486.0,107563.0
4,08015,Badalona,2025,88146.0,15586.0,103732.0
5,08252,Barberà del Vallès,2021,14535.0,1918.0,16453.0
6,08252,Barberà del Vallès,2022,15027.0,1516.0,16543.0
7,08252,Barberà del Vallès,2023,15296.0,1418.0,16714.0
8,08252,Barberà del Vallès,2024,15576.0,1408.0,16984.0
9,08252,Barberà del Vallès,2025,14535.0,1918.0,16453.0


activity_analysis: rows=235, municipalities=47, invalid=235
REVIEW REQUIRED — distance_analysis: invalid codes


,municipality_code,municipality,distance_to_barcelona_km,distance_to_girona_km,distance_to_tarragona_km,distance_to_lleida_km
0,08015,Badalona,11.9809,90.4004,106.2319,160.3240
1,08019,Barcelona,0.0000,102.6975,96.3642,156.0042
2,08056,Castelldefels,23.5322,120.6056,76.2651,157.8519
3,08073,Cornellà de Llobregat,14.6353,107.8245,85.9134,147.4897
4,08077,Esplugues de Llobregat,9.7772,105.0115,89.8806,148.6391
5,08089,Gavà,22.4092,119.4827,78.6702,160.2571
6,08096,Granollers,32.6344,74.4900,118.7883,164.6437
7,08101,"Hospitalet de Llobregat, l'",11.5371,107.3437,88.6893,149.1633
8,08102,Igualada,68.1386,143.1740,89.9044,93.4865
9,08113,Manresa,64.9334,118.0991,113.1706,112.9640


distance_analysis: rows=47, municipalities=47, invalid=47
Datasets exportados correctamente:

analysis_universe.csv: 0 invalid codes out of 47 rows
population_analysis.csv: 0 invalid codes out of 564 rows
sales_analysis.csv: 0 invalid codes out of 564 rows
rent_analysis.csv: 0 invalid codes out of 564 rows
income_analysis.csv: 0 invalid codes out of 517 rows
demographic_analysis.csv: 0 invalid codes out of 235 rows
activity_analysis.csv: 0 invalid codes out of 235 rows
distance_analysis.csv: 0 invalid codes out of 47 rows
